<a href="https://colab.research.google.com/github/71percentbanana/gridathon/blob/main/Gridlock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install catboost -q
!pip install pygeohash -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.0 MB/s eta 0:00:00


In [11]:
import pandas as pd
import numpy as np
import pygeohash as pgh

from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score



df = pd.read_csv("https://raw.githubusercontent.com/71percentbanana/gridathon/refs/heads/main/train.csv")
test_df = pd.read_csv("https://raw.githubusercontent.com/71percentbanana/gridathon/refs/heads/main/test.csv")


for data in [df, test_df]:
    data['Temperature'] = data['Temperature'].fillna(df['Temperature'].median())
    data['RoadType'] = data['RoadType'].fillna('Unknown')
    data['Weather'] = data['Weather'].fillna('Unknown')


for data in [df, test_df]:
    data[['hour', 'minute']] = data['timestamp'].str.split(':', expand=True)

    data['hour'] = data['hour'].astype(int)
    data['minute'] = data['minute'].astype(int)

    data['hour_sin'] = np.sin(2 * np.pi * data['hour'] / 24)
    data['hour_cos'] = np.cos(2 * np.pi * data['hour'] / 24)

    data['minute_sin'] = np.sin(2 * np.pi * data['minute'] / 60)
    data['minute_cos'] = np.cos(2 * np.pi * data['minute'] / 60)

    data['is_weekend'] = (data['day'] >= 5).astype(int)


for data in [df, test_df]:
    data['RoadType_Lanes'] = (
        data['RoadType'].astype(str) + "_" +
        data['NumberofLanes'].astype(str)
    )



for data in [df, test_df]:
    data['geohash_4'] = data['geohash'].str[:4]
    data['geohash_5'] = data['geohash'].str[:5]
    data['geohash_6'] = data['geohash'].str[:6]

    data['latitude'] = data['geohash'].apply(lambda x: pgh.decode(x)[0])
    data['longitude'] = data['geohash'].apply(lambda x: pgh.decode(x)[1])


features = [
    'geohash_4',
    'geohash_5',
    'geohash_6',
    'latitude',
    'longitude',
    'day',
    'is_weekend',
    'RoadType',
    'NumberofLanes',
    'LargeVehicles',
    'Landmarks',
    'Temperature',
    'Weather',
    'hour_sin',
    'hour_cos',
    'minute_sin',
    'minute_cos',
    'RoadType_Lanes'

]

X = df[features]
y = df['demand']
test_X = test_df[features]

hour_mean = df.groupby('hour')['demand'].mean()

df['hour_avg_demand'] = df['hour'].map(hour_mean)
test_df['hour_avg_demand'] = test_df['hour'].map(hour_mean)

features.append('hour_avg_demand')
X = df[features]
y = df['demand']

test_X = test_df[features]

cat_features = [
    'geohash_4',
    'geohash_5',
    'geohash_6',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'RoadType_Lanes'
]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

seeds = [42, 123, 2025]

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(test_X))


for seed in seeds:

    print(f"\nTraining Seed {seed}")

    seed_oof = np.zeros(len(X))
    seed_test = np.zeros(len(test_X))

    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):

        print(f"Fold {fold + 1}")

        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = CatBoostRegressor(
            iterations=1500,
            depth=8,
            learning_rate=0.05,
            loss_function='RMSE',
            random_seed=seed,
            l2_leaf_reg=5,
            bagging_temperature=0.7,
            verbose=0,
            early_stopping_rounds=100
        )

        model.fit(
            X_train,
            y_train,
            cat_features=cat_features
        )

        val_preds = model.predict(X_val)

        seed_oof[val_idx] = val_preds

        seed_test += model.predict(test_X) / 5

    oof_preds += seed_oof / len(seeds)
    test_preds += seed_test / len(seeds)


rmse = np.sqrt(mean_squared_error(y, oof_preds))
r2 = r2_score(y, oof_preds)

print("\nFINAL RESULTS")
print("OOF RMSE:", rmse)
print("OOF R2:", r2)

importance = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
})

print(
    importance.sort_values(
        by='Importance',
        ascending=False
    )
)


test_preds = np.clip(test_preds, 0, 1)

submission = pd.DataFrame({
    'Index': test_df['Index'],
    'demand': test_preds
})

submission.to_csv("submission.csv", index=False)

print(submission.head())


Training Seed 42
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

Training Seed 123
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

Training Seed 2025
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5

FINAL RESULTS
OOF RMSE: 0.031343153653862386
OOF R2: 0.9514096890921266
            Feature  Importance
17   RoadType_Lanes   27.967580
7          RoadType   18.465551
2         geohash_6   13.224064
9     LargeVehicles    8.058781
1         geohash_5    5.331356
18  hour_avg_demand    4.837964
8     NumberofLanes    4.608447
4         longitude    4.105003
3          latitude    3.913153
13         hour_sin    3.035777
14         hour_cos    2.886376
0         geohash_4    1.426344
5               day    1.275386
11      Temperature    0.277356
12          Weather    0.222066
16       minute_cos    0.190291
15       minute_sin    0.117557
10        Landmarks    0.056947
6        is_weekend    0.000000
   Index    demand
0      0  0.052528
1      1  0.034311
2      2  0.016811
3      3  0.040059
4      4  0.062245
